In [0]:
from pyspark.sql.functions import col, explode

source_path = (
    "abfss://artgulidov@dlsua5816bd.dfs.core.windows.net/"
    "streaming/"
)

schema_path = (
    "abfss://artgulidov@dlsua5816bd.dfs.core.windows.net/streaming/streaming_metadata/earthquake_schema/"
)

checkpoint_path = (
    "abfss://artgulidov@dlsua5816bd.dfs.core.windows.net/streaming/streaming_metadata/earthquake_checkpoint/"
)

earthquake_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.maxFilesPerTrigger", 1)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(source_path)
)

earthquakes = (
    earthquake_stream
    .select(explode("features").alias("feature"))
    .select(
        col("feature.id").alias("earthquake_id"),
        col("feature.properties.mag").alias("magnitude"),
        col("feature.properties.place").alias("place"),
        col("feature.properties.time").alias("event_time"),
        col("feature.properties.tsunami").alias("tsunami"),
        col("feature.properties.alert").alias("alert"),
        col("feature.geometry.coordinates")[0].alias("longitude"),
        col("feature.geometry.coordinates")[1].alias("latitude"),
        col("feature.geometry.coordinates")[2].alias("depth")
    )
)

query = (
    earthquakes.writeStream
        .format("delta")
        .outputMode("append")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .option(
            "checkpointLocation",
            checkpoint_path
        )
        .toTable("dbr_dev_ua5816bd.gulidov_artem_staging.streaming_earthquakes_staging")
)